Konteks / Skenario
Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun dashboard pemantauan pesanan real-time untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (prototype) pipeline streaming yang mensimulasikan kebutuhan ini.

In [1]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas10_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Menyiapkan folder simulasi streaming
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

print("SparkSession siap dan folder dibuat:", folder_tugas)

26/11/05 17:06:19 WARN Utils: Your hostname, adnan resolves to a loopback address: 127.0.1.1; using 10.90.69.81 instead (on interface wlp2s0)
26/11/05 17:06:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 17:06:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap dan folder dibuat: tugas10_stream


A. Menyiapkan Streaming Source

In [2]:
# 1. Definisi skema eksplisit
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# 2. Definisi readStream
df_stream = (
    spark.readStream
    .schema(skema_pesanan)
    .csv(folder_tugas)
)

B. Agregasi per Cabang 

In [3]:
import pyspark.sql.functions as F

# Tambah kolom turunan total_harga
df_transformed = df_stream.withColumn("total_harga", F.col("qty") * F.col("harga_satuan"))

# Buat agregasi groupBy per cabang
query_cabang = (
    df_transformed.groupBy("nama_cabang")
    .agg(
        F.sum("total_harga").alias("total_pendapatan"),
        F.count("pesanan_id").alias("jumlah_pesanan")
    )
    .writeStream
    .format("memory")
    .queryName("agregasi_cabang")
    .outputMode("complete")
    .start()
)

C. Agregasi Berbasis Window Waktu

In [4]:
# Buat streaming query kedua dengan window waktu 2 menit
query_waktu = (
    df_transformed
    .groupBy(
        F.window(F.col("waktu_pesan"), "2 minutes"),
        F.col("nama_cabang")
    )
    .agg(F.sum("total_harga").alias("total_pendapatan_window"))
    .writeStream
    .format("memory")
    .queryName("agregasi_waktu")
    .outputMode("update")
    .start()
)

D. Menguji dengan Data Simulasi

In [5]:
import time
import csv

# --- BATCH 1 ---
file1 = os.path.join(folder_tugas, "batch1.csv")
with open(file1, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["P001", "Cabang Sudirman", "Kopi Susu", 2, 20000, "2026-06-07 10:00:00"])
    writer.writerow(["P002", "Cabang Sudirman", "Roti Bakar", 1, 15000, "2026-06-07 10:01:00"])
    writer.writerow(["P003", "Cabang Kemang", "Americano", 3, 25000, "2026-06-07 10:00:30"])

time.sleep(5)
print("--- Hasil Batch 1 ---")
spark.sql("SELECT * FROM agregasi_cabang").show(truncate=False)

# --- BATCH 2 ---
file2 = os.path.join(folder_tugas, "batch2.csv")
with open(file2, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["P004", "Cabang Sudirman", "Kopi Susu", 1, 20000, "2026-06-07 10:02:10"])
    writer.writerow(["P005", "Cabang Menteng", "Latte", 2, 30000, "2026-06-07 10:01:15"])

time.sleep(5)
print("--- Hasil Batch 2 ---")
spark.sql("SELECT * FROM agregasi_cabang").show(truncate=False)

# --- BATCH 3 ---
file3 = os.path.join(folder_tugas, "batch3.csv")
with open(file3, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["P006", "Cabang Kemang", "Cappuccino", 2, 28000, "2026-06-07 10:03:00"])
    writer.writerow(["P007", "Cabang Menteng", "Espresso", 4, 18000, "2026-06-07 10:02:40"])

time.sleep(5)
print("--- Hasil Batch 3 ---")
spark.sql("SELECT * FROM agregasi_cabang").show(truncate=False)

--- Hasil Batch 1 ---
+---------------+----------------+--------------+
|nama_cabang    |total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Sudirman|55000           |2             |
|Cabang Kemang  |75000           |1             |
+---------------+----------------+--------------+

--- Hasil Batch 2 ---
+---------------+----------------+--------------+
|nama_cabang    |total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Sudirman|75000           |3             |
|Cabang Kemang  |75000           |1             |
|Cabang Menteng |60000           |1             |
+---------------+----------------+--------------+

--- Hasil Batch 3 ---
+---------------+----------------+--------------+
|nama_cabang    |total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Sudirman|75000           |3             |
|Cabang Kemang  |131000          |2             |
|Cabang Menteng |132000         

E. Sink Berbasis Berkas (Tantangan)

In [6]:
# 1. Jalankan query ketiga (sink parquet)
query3 = (
    df_stream.writeStream
    .format("parquet")
    .option("path", "tugas10_output")
    .option("checkpointLocation", "tugas10_checkpoint")
    .outputMode("append")
    .start()
)

# 2. Kirim satu batch data baru
file_baru = os.path.join(folder_tugas, "batch_baru.csv")
with open(file_baru, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["P008", "Cabang Sudirman", "Manual Brew", 1, 35000, "2026-06-07 10:05:00"])

# Tunggu beberapa detik agar tersimpan ke format parquet
time.sleep(6)

# 3. Buktikan berkas tersimpan menggunakan perintah shell / membaca ulang parquet
print("--- Daftar file di folder tugas10_output ---")
!ls tugas10_output

print("\n--- Membaca kembali sebagai DataFrame batch biasa ---")
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.show(truncate=False)

--- Daftar file di folder tugas10_output ---
part-00000-35e28ae1-a592-42a4-ad70-ba93a083a791-c000.snappy.parquet
part-00001-7bf5399e-a470-4ed5-8b97-768e14a96df7-c000.snappy.parquet
part-00002-666fec5b-cf87-4e0f-9f7f-478c10fd9b3f-c000.snappy.parquet
part-00003-3b173533-d280-4e4a-b7a9-defd5a1898be-c000.snappy.parquet
_spark_metadata

--- Membaca kembali sebagai DataFrame batch biasa ---
+----------+---------------+-----------+---+------------+-------------------+
|pesanan_id|nama_cabang    |nama_menu  |qty|harga_satuan|waktu_pesan        |
+----------+---------------+-----------+---+------------+-------------------+
|P008      |Cabang Sudirman|Manual Brew|1  |35000       |2026-06-07 10:05:00|
|P001      |Cabang Sudirman|Kopi Susu  |2  |20000       |2026-06-07 10:00:00|
|P002      |Cabang Sudirman|Roti Bakar |1  |15000       |2026-06-07 10:01:00|
|P003      |Cabang Kemang  |Americano  |3  |25000       |2026-06-07 10:00:30|
|P006      |Cabang Kemang  |Cappuccino |2  |28000       |2026-06-0

In [ ]:
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()